In [84]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
df=pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.sample(10)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
1987,5419-KLXBN,Female,0,Yes,Yes,25,Yes,No,Fiber optic,No,...,Yes,No,Yes,No,Month-to-month,Yes,Bank transfer (automatic),89.15,2257.75,Yes
1988,3424-NMNBO,Male,1,Yes,No,58,Yes,Yes,Fiber optic,Yes,...,Yes,No,Yes,Yes,One year,Yes,Electronic check,108.85,6287.25,Yes
3914,1755-FZQEC,Male,0,No,No,39,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,No,Mailed check,19.90,791.15,No
3928,4647-MUZON,Female,0,Yes,No,18,Yes,No,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Credit card (automatic),95.95,1745.5,No
2759,1194-SPVSP,Male,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Bank transfer (automatic),19.65,19.65,No
2999,1038-RQOST,Male,0,Yes,Yes,19,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.60,414.95,No
5125,2982-VPSGI,Female,0,Yes,No,11,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,94.00,1078.9,Yes
6906,9945-PSVIP,Female,0,Yes,Yes,25,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Mailed check,18.70,383.65,No
381,6905-NIQIN,Male,0,No,No,1,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,No,Mailed check,50.65,50.65,Yes
1058,2074-GKOWZ,Male,0,Yes,Yes,2,Yes,No,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),89.55,185.55,Yes


In [86]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [87]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [88]:
df.drop("customerID", axis=1, inplace=True)

In [89]:
df["TotalCharges"].dtypes

dtype('O')

In [90]:
# it is numeric bt dataset shows as object . so convert it into numeric

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [91]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Churn"] = le.fit_transform(df["Churn"])

In [92]:
X=df.drop("Churn",axis=1)
y=df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [93]:
# FEATURE ENGINEERING

num_col=X.select_dtypes(include=np.number).columns
cat_col=X.select_dtypes(exclude=np.number).columns
print(num_col)
print(cat_col)

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')
Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')


In [94]:
#feature engineering
from sklearn.pipeline import Pipeline


num_pipeline=Pipeline(
      steps=[
          ("imputer", SimpleImputer(strategy="median")),
          ("scaler", StandardScaler())
      ]
)

cat_pipeline=Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder())
    ]
)
num_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [95]:
cat_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder())])

In [96]:
prepocessing=ColumnTransformer(
  transformers=[
      ("num", num_pipeline, num_col),
      ("cat", cat_pipeline, cat_col)
  ]
)

prepocessing



ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder', OneHotEncoder())]),
                                 Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])

In [97]:
# modal implement
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
lr=Pipeline(
    steps=[
        ("prepocessing", prepocessing),
        ("model", LogisticRegression())
    ]
)

lr.fit(X_train, y_train)


#Train acuracy
y_pred_train=lr.predict(X_train)
print("logistic regresssion train \n")



#Test acuracy


y_pred_test=lr.predict(X_test)

print("\nlogistic regresssion test \n")

accuracy=accuracy_score(y_test,y_pred_test)
pecision=precision_score(y_test,y_pred_test)
recall=recall_score(y_test,y_pred_test)
f1=f1_score(y_test,y_pred_test)

print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)


logistic regresssion train 


logistic regresssion test 

accuracy 0.8055358410220014
pecision 0.6572327044025157
recall 0.5588235294117647
f1 0.6040462427745664


In [98]:
# random forest hyperperameter tunning


from sklearn.ensemble import RandomForestClassifier

random_forest=Pipeline(
    steps=[
        ("prepocessing", prepocessing),
        ("model", RandomForestClassifier(
               n_estimators=300,
               max_depth=10,
               min_samples_split=5,
               min_samples_leaf=2,
               random_state=42))
    ]
)

random_forest.fit(X_train, y_train)

# train part
y_pred_train=random_forest.predict(X_train)

accuracy=accuracy_score(y_train,y_pred_train)
pecision=precision_score(y_train,y_pred_train)
recall=recall_score(y_train,y_pred_train)
f1=f1_score(y_train,y_pred_train)
print("Random Forest train part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)

# test accauracy
y_pred_test=random_forest.predict(X_test)

accuracy=accuracy_score(y_test,y_pred_test)
pecision=precision_score(y_test,y_pred_test)
recall=recall_score(y_test,y_pred_test)

f1=f1_score(y_test,y_pred_test)
print("\nRandom Forest test part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)


# here without hyper peramter random forest also overfitted result shows  n_estimators=300,max_depth=10, min_samples_split=5,min_samples_leaf=2,random_state=42


# this perameter handles the overfitting condition


Random Forest train part analysis

accuracy 0.8601348952786653
pecision 0.7895167895167895
recall 0.6448160535117057
f1 0.7098674521354934

Random Forest test part analysis

accuracy 0.8055358410220014
pecision 0.6700680272108843
recall 0.5267379679144385
f1 0.5898203592814372


In [99]:

# decision tree classifier
from sklearn.tree import DecisionTreeClassifier

dt=Pipeline(
    steps=[
        ("prepocessing",prepocessing),
        ("model",DecisionTreeClassifier(random_state=42))

    ]

)

dt.fit(X_train, y_train)
# train part
y_pred_train=dt.predict(X_train)

accuracy=accuracy_score(y_train,y_pred_train)
pecision=precision_score(y_train,y_pred_train)
recall=recall_score(y_train,y_pred_train)
f1=f1_score(y_train,y_pred_train)
print("Decsion Tree Classifier train part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)

# test accauracy
y_pred_test=dt.predict(X_test)

accuracy=accuracy_score(y_test,y_pred_test)
pecision=precision_score(y_test,y_pred_test)
recall=recall_score(y_test,y_pred_test)

f1=f1_score(y_test,y_pred_test)
print("\nDecsion Tree Classifier test part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)


# heavy overfitting


Decsion Tree Classifier train part analysis

accuracy 0.9980475683351083
pecision 0.9993270524899058
recall 0.9933110367892977
f1 0.996309963099631

Decsion Tree Classifier test part analysis

accuracy 0.7288857345635202
pecision 0.4896373056994819
recall 0.5053475935828877
f1 0.49736842105263157


In [100]:

# decision tree classifier with  perameter tunning

from sklearn.tree import DecisionTreeClassifier

dt=Pipeline(
    steps=[
        ("prepocessing",prepocessing),
        ("model",DecisionTreeClassifier(
                         max_depth=5,
                         min_samples_split=10,
                         min_samples_leaf=5,
                         random_state=42))

    ]

)

dt.fit(X_train, y_train)
# train part
y_pred_train=dt.predict(X_train)

accuracy=accuracy_score(y_train,y_pred_train)
pecision=precision_score(y_train,y_pred_train)
recall=recall_score(y_train,y_pred_train)
f1=f1_score(y_train,y_pred_train)
print("Decsion Tree Classifier train part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)

# test accauracy
y_pred_test=dt.predict(X_test)

accuracy=accuracy_score(y_test,y_pred_test)
pecision=precision_score(y_test,y_pred_test)
recall=recall_score(y_test,y_pred_test)

f1=f1_score(y_test,y_pred_test)
print("\nDecsion Tree Classifier test part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)

Decsion Tree Classifier train part analysis

accuracy 0.8031593894213702
pecision 0.651017214397496
recall 0.5565217391304348
f1 0.6000721240533718

Decsion Tree Classifier test part analysis

accuracy 0.7984386089425124
pecision 0.6347305389221557
recall 0.5668449197860963
f1 0.5988700564971752


In [103]:
# KNN with grid search
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

knn_with_grid = Pipeline(
    steps=[
        ("prepocessing", prepocessing),
        ("model", KNeighborsClassifier())
    ]
)


grid_param={
    "model__n_neighbors":[5,10,15,20,25],
    "model__weights":["uniform","distance"],
    "model__p":[1,2]
}


grid_search = GridSearchCV(
    estimator=knn_with_grid,
    param_grid=grid_param,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)


grid_search.fit(X_train, y_train)

#train accuracy part
y_pred_train = grid_search.predict(X_train)
accuracy=accuracy_score(y_train,y_pred_train)
pecision=precision_score(y_train,y_pred_train)
recall=recall_score(y_train,y_pred_train)
f1=f1_score(y_train,y_pred_train)

print("  KNN with gridsearch train part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)


# test accuracy
y_pred_test=dt.predict(X_test)
accuracy=accuracy_score(y_test,y_pred_test)
pecision=precision_score(y_test,y_pred_test)
recall=recall_score(y_test,y_pred_test)
f1=f1_score(y_test,y_pred_test)


print("\n KNN with gridsearch test part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)

  KNN with gridsearch train part analysis

accuracy 0.8130990415335463
pecision 0.671583850931677
recall 0.5785953177257525
f1 0.6216313330937837

 KNN with gridsearch test part analysis

accuracy 0.7984386089425124
pecision 0.6347305389221557
recall 0.5668449197860963
f1 0.5988700564971752


| Model                       |   Accuracy |  Precision |     Recall |   F1 Score | Overfitting |
| --------------------------- | ---------: | ---------: | ---------: | ---------: | ----------- |
| **Logistic Regression**     | **80.55%** |     65.72% |     55.88% | **60.40%** | ❌ No        |
| **Random Forest (Tuned)**   | **80.55%** | **67.01%** |     52.67% |     58.98% | ⚠️ Slight   |
| **Decision Tree (Untuned)** |     72.89% |     48.96% |     50.53% |     49.74% | ❌ Heavy     |
| **Decision Tree (Tuned)**   |     79.84% |     63.47% | **56.68%** |     59.89% | ✅ No        |
| **KNN + GridSearch**        |     79.84% |     63.47% | **56.68%** |     59.89% | ✅ No        |


After comparing all the classification models, I found that Logistic Regression performed the best on the Telco Customer Churn dataset. It achieved an accuracy of 80.55% and the highest F1 score of 60.4%. Another reason I selected this model is that its training and testing results were very close, which means it generalized well and did not overfit the data. Although Random Forest achieved the same accuracy and slightly better precision, its recall and F1 score were lower than Logistic Regression. The Decision Tree initially suffered from overfitting, but after hyperparameter tuning its performance improved significantly. Overall, considering accuracy, F1 score, and model stability, I believe Logistic Regression is the most suitable model for this dataset.